In [1]:
!pip install pycountry

In [2]:
import wbdata
import pandas as pd
import datetime
import os
import pycountry

Key '-2148729463605886678' not in persistent cache.
Key '7374717871831342371' not in persistent cache.
Key '-3862665431376555150' not in persistent cache.
Key '-7552956687330324976' not in persistent cache.
Key '346127621069866508' not in persistent cache.
Key '-854483118462797963' not in persistent cache.
Key '-9015870295102693135' not in persistent cache.
Key '-8289753835734040624' not in persistent cache.
Key '-5768663902387138238' not in persistent cache.
Key '-2957660514869852271' not in persistent cache.
Key '-2249594387839253969' not in persistent cache.
Key '5649489920560937515' not in persistent cache.
Key '50521798043904916' not in persistent cache.
Key '-3712507995611622128' not in persistent cache.
Key '-5429610752402375836' not in persistent cache.
Key '-7306878202039841578' not in persistent cache.
Key '-7143546720173044308' not in persistent cache.
Key '7612906097582737482' not in persistent cache.
Key '4111275620908234175' not in persistent cache.
Key '-4135364407259287

In [3]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
PROCESSED_DATA_DIR_PATH = os.path.join(DATA_DIR_PATH, "processed_data")

In [4]:


# 1. Define the indicators you want to fetch
indicators = {
    'NY.GDP.MKTP.KD':       'gdp_2015_usd',
    # 'NY.GDP.PCAP.KD':       'gdp_per_capita',
    'SP.POP.TOTL':          'population',
    # 'SP.POP.GROW':          'population_growth',
    'EG.USE.PCAP.KG.OE':    'energy_per_capita',
    'EG.FEC.RNEW.ZS':       'renewable_share',
    'EG.GDP.PUSE.KO.PP.KD': 'energy_intensity',
    'SP.URB.TOTL.IN.ZS':    'urbanization_rate',
    'NV.IND.TOTL.ZS':       'industry_size',
    'AG.LND.FRST.ZS':       'forest_area'
}

date_range = (datetime.datetime(2000,1,1), datetime.datetime(2022,1,1))

df = wbdata.get_dataframe(
    indicators,
    country="all",
    date=date_range,
    freq='Y',
    parse_dates=True
).reset_index().rename(columns={'country':'region','date':'year'})

# 2) Pull the country lookup (wbdata.get_countries returns a list of dicts)
country_list = wbdata.get_countries()  

#  Inspect one entry to see its keys:
# print(country_list[0])
# -> {'id': 'ABW', 'iso2Code': 'AW', 'name': 'Aruba', …}

# 3) Build a small DataFrame of name → ISO-2
country_df = pd.DataFrame(country_list)[['name','iso2Code']]
country_df.columns = ['region','iso2']  

# 4) Merge it onto your indicators DataFrame
df = df.merge(country_df, on='region', how='left')

# 5) (Optional) Convert ISO-2 → ISO-3 with pycountry
def iso2_to_iso3(c2):
    try:
        return pycountry.countries.get(alpha_2=c2).alpha_3
    except:
        return None

df['iso_alpha_3'] = df['iso2'].map(iso2_to_iso3)

# make year only the year
df['year'] = df['year'].dt.year

# drop iso2 and region
df = df.drop(columns=['iso2', 'region'])


In [5]:
df

,year,gdp_2015_usd,population,energy_per_capita,renewable_share,energy_intensity,urbanization_rate,industry_size,forest_area,iso_alpha_3
0,2022,1.040350e+12,731821393.0,565.488909,NaN,7.303970,37.909012,26.921919,29.737205,None
1,2021,1.004646e+12,713090928.0,570.998888,NaN,7.156658,37.393633,26.075267,29.955194,None
2,2020,9.606813e+11,694446100.0,563.976201,66.123449,7.107927,36.884034,25.434235,30.174252,None
3,2019,9.890095e+11,675950189.0,586.441491,63.387090,7.212707,36.384272,26.421751,30.391626,None
4,2018,9.677734e+11,657801085.0,583.763039,62.242631,7.273489,35.893398,27.859732,30.611512,None
...,...,...,...,...,...,...,...,...,...,...
6113,2004,1.433634e+10,12365896.0,449.272022,81.800000,6.296510,34.294000,24.382749,46.999354,ZWE
6114,2003,1.522027e+10,12232323.0,477.200797,78.000000,6.362219,34.479000,NaN,47.118444,ZWE
6115,2002,1.833658e+10,12087653.0,519.581800,74.700000,7.123918,34.585000,NaN,47.237534,ZWE
6116,2001,2.012665e+10,11971901.0,540.270614,72.300000,7.592652,34.170000,NaN,47.356624,ZWE


In [6]:
df.to_csv(os.path.join(PROCESSED_DATA_DIR_PATH, 'wb_control_vars.csv'), index=False)